In [ ]:
# 1. Install dependencies (if not already installed)
!pip install google-cloud-texttospeech


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.1/200.1 kB 7.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
data = pd.read_csv("data.csv")
mundari = data["MUNDARI"] # or bhili
mundari

,MUNDARI
0,सीता अर राजू बरनकिन अकिनअः सोंगे कोलोः मिसाते ...
1,एंगाते ककला जअः किना सीता ऐ सीता ..राजू ऐ राजू...
2,बरन बोकोबोया एंगाते तअः किन निर केदा।
3,"आजा हड़म दरूकोरे दअःए दुलतना, जियङ बुड़िया चउल..."
4,अपुते साइकिलते पिटिःते सेनोः तना।
...,...
1817,सूची बइएपे।
1818,रबङ रूआ एंगा एनोफिल सिकिड़ि हुअः ते होबाओ:आ।
1819,दंडाएपे आद ओलेपे : -
1820,मोटोर कोरे का दुब लगातिङआ।


In [ ]:
import os
from tqdm import tqdm
from google.cloud import texttospeech



creds_path = "<creds_path>"
client = texttospeech.TextToSpeechClient.from_service_account_file(creds_path)
output_dir = "/out"

def synthesize_and_save(text, filename):
    input_text = texttospeech.SynthesisInput(text=str(text))

    voice = texttospeech.VoiceSelectionParams(
        language_code="hi-IN",
        name="hi-IN-Chirp3-HD-Achird"  # Change if needed
    )

    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.LINEAR16,
        speaking_rate=0.8
    )

    response = client.synthesize_speech(
        input=input_text,
        voice=voice,
        audio_config=audio_config
    )

    path = os.path.join(output_dir, filename)
    with open(path, "wb") as out:
        out.write(response.audio_content)
    return path

# 6. Loop through dataset with tqdm

start_index = 1

for i, text in tqdm(enumerate(mundari[start_index-1:], start=start_index),
                    total=len(mundari) - (start_index-1),
                    desc="Generating Audio"):
    if not isinstance(text, str) or len(text) > 270:
        continue
    filename = os.path.join(output_dir, f"mun_audio{i}.wav")
    synthesize_and_save(text, filename)


Generating Audio: 100%|██████████| 1822/1822 [24:53<00:00,  1.22it/s]


In [ ]:
import os
import wave

# 🔹 Change this to the full path of your folder
folder_path = "/out"

# Counters
less_5 = 0
between_5_10 = 0
greater_10 = 0

# Iterate over wav files
for filename in os.listdir(folder_path):
    if filename.lower().endswith(".wav"):
        file_path = os.path.join(folder_path, filename)
        try:
            with wave.open(file_path, 'rb') as wf:
                frames = wf.getnframes()
                rate = wf.getframerate()
                duration = frames / float(rate)

                if duration < 5:
                    less_5 += 1
                elif 5 <= duration <= 15:
                    between_5_10 += 1
                else:
                    greater_10 += 1
        except Exception as e:
            print(f"Error reading {filename}: {e}")

print("Files with duration < 5 sec:", less_5)
print("Files with duration 5–10 sec:", between_5_10)
print("Files with duration > 10 sec:", greater_10)


Files with duration < 5 sec: 731
Files with duration 5–10 sec: 1057
Files with duration > 10 sec: 34


In [ ]:
import shutil
shutil.make_archive("/out", 'zip', "/out")


In [ ]:
dataset = []

In [ ]:
df

,audio,MUNDARI
0,out_16k_main/mun_audio1.wav,सीता अर राजू बरनकिन अकिनअः सोंगे कोलोः मिसाते ...
1,out_16k_main/mun_audio10.wav,एंगाते ककला जअः किना सीता ऐ सीता ..राजू ऐ राजू...
2,out_16k_main/mun_audio100.wav,बरन बोकोबोया एंगाते तअः किन निर केदा।
3,out_16k_main/mun_audio1000.wav,"आजा हड़म दरूकोरे दअःए दुलतना, जियङ बुड़िया चउल..."
4,out_16k_main/mun_audio1001.wav,अपुते साइकिलते पिटिःते सेनोः तना।
...,...,...
1817,out_16k_main/mun_audio995.wav,सूची बइएपे।
1818,out_16k_main/mun_audio996.wav,रबङ रूआ एंगा एनोफिल सिकिड़ि हुअः ते होबाओ:आ।
1819,out_16k_main/mun_audio997.wav,दंडाएपे आद ओलेपे : -
1820,out_16k_main/mun_audio998.wav,मोटोर कोरे का दुब लगातिङआ।


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict, Audio
import os

# Paths
csv_path = "data.csv"
audio_folder = "out_16k_main/"

# Load CSV
df = pd.read_csv(csv_path)

# Sort audio files (important!)
audio_files = sorted(os.listdir(audio_folder))

# Full paths
audio_paths = [os.path.join(audio_folder, f) for f in audio_files]

# Match by index
df = df.iloc[:len(audio_paths)]  # safety check
df["audio"] = audio_paths

# Keep only required columns
df = df[["audio", "MUNDARI"]]

# Convert to HF dataset
dataset = Dataset.from_pandas(df)

# Cast audio column
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Split: last 100 → test
train_dataset = dataset.select(range(len(dataset) - 100))
test_dataset = dataset.select(range(len(dataset) - 100, len(dataset)))

dataset_d = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(dataset_d)

DatasetDict({
    train: Dataset({
        features: ['audio', 'MUNDARI'],
        num_rows: 1722
    })
    test: Dataset({
        features: ['audio', 'MUNDARI'],
        num_rows: 100
    })
})


In [ ]:
from IPython.display import Audio

sample = dataset_dict["train"][0]

Audio(
    data=sample["audio"]["array"],
    rate=sample["audio"]["sampling_rate"]
)

In [ ]:
import soundfile as sf
output_dir = "out_16k_main"
data, sr = sf.read(os.path.join(output_dir, audio_files[0]))
print(sr)   # should be 16000

16000
